In [1]:
import os
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz
import warnings
warnings.filterwarnings("ignore")
import time
from datetime import datetime, time as dtime, timedelta
from zoneinfo import ZoneInfo

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
CACHE_DIR    = "/content/drive/MyDrive/stocks"
RESULTS_DIR  = "/content/drive/MyDrive/stock_screener_results"
HISTORY_FILE_PATH = os.path.join(CACHE_DIR, "stock_history.csv")

os.makedirs(CACHE_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

NY_TZ = pytz.timezone("America/New_York")
LINE  = "-" * 72
DLINE = "=" * 72

In [4]:
url_1 = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/refs/heads/main/amex/amex_full_tickers.json"
r_1 = requests.get(url_1)

url_2 = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/refs/heads/main/nasdaq/nasdaq_full_tickers.json"
r_2 = requests.get(url_2)

url_3 = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/refs/heads/main/nyse/nyse_full_tickers.json"
r_3 = requests.get(url_3)

In [5]:
json_1 = r_1.json()
json_1[0].keys()

json_2 = r_2.json()
json_2[0].keys()

json_3 = r_3.json()
json_3[0].keys()

dict_keys(['symbol', 'name', 'lastsale', 'netchange', 'pctchange', 'volume', 'marketCap', 'country', 'ipoyear', 'industry', 'sector', 'url'])

In [6]:
combined_json = json_1 + json_2 + json_3
all_symbols_df = pd.DataFrame(combined_json)

print(f"Total unique symbols: {all_symbols_df['symbol'].nunique()}")

df_const = all_symbols_df[['symbol','name','sector','industry','country','marketCap']]

df_const = df_const.rename(columns={'sector':'GICS Sector','industry':'GICS Sub-Industry','symbol':'Symbol','name':'Security'})

df_const['marketCap'] = pd.to_numeric(df_const['marketCap'], errors='coerce')
df_const.dropna(subset=['marketCap'], inplace=True)
df_const['marketCap'] = df_const['marketCap'].astype(int)
df_const["Symbol"] = df_const["Symbol"].str.replace("/", ".", regex=False)
df_const["Symbol"] = df_const["Symbol"].str.replace("-", ".", regex=False)
display(df_const.head())

Total unique symbols: 7232


,Symbol,Security,GICS Sector,GICS Sub-Industry,country,marketCap
0,ACCS,ACCESS Newswire Inc. Common Stock,Consumer Discretionary,Professional Services,United States,12363891
1,ACU,Acme United Corporation. Common Stock,Consumer Discretionary,Industrial Machinery/Components,United States,278337312
2,AEF,abrdn Emerging Markets ex-China Fund Inc. Comm...,Finance,Finance/Investors Services,United States,874467563
3,AEON,AEON Biopharma Inc. Class A Common Stock,Health Care,Biotechnology: Pharmaceutical Preparations,United States,12365293
4,AGIG,Abundia Global Impact Group Inc. Common stock,Energy,Oil & Gas Production,United States,35257838


In [7]:
df_const['marketCap'] = df_const['marketCap'] / 1000000000
df_const = df_const[df_const['marketCap'] >= 5]
print(f"Total unique symbols: {df_const['Symbol'].nunique()}")
display(df_const.head())

Total unique symbols: 1441


,Symbol,Security,GICS Sector,GICS Sub-Industry,country,marketCap
34,BTG,B2Gold Corp Common shares (Canada),Basic Materials,Precious Metals,Canada,6.815751
40,CET,Central Securities Corporation Common Stock,Finance,Finance/Investors Services,United States,13.744611
83,EQX,Equinox Gold Corp. Common Shares,Basic Materials,Precious Metals,Canada,12.918361
132,IMO,Imperial Oil Limited Common Stock,Energy,Integrated oil Companies,Canada,59.714212
309,AAL,American Airlines Group Inc. Common Stock,Consumer Discretionary,Air Freight/Delivery Services,United States,9.342920


In [8]:
df_const.to_csv('ticker_data.csv')
!cp ticker_data.csv "/content/drive/MyDrive/stocks/ticker_data.csv"

In [9]:
col = "Symbol" if "Symbol" in df_const.columns else df_const.columns[0]
symbols = df_const[col].tolist()

filtered_symbols = [
    s.replace(".", "-") for s in symbols
    if len(s) <= 5 and not s.startswith("^")
]

print(f"Successfully loaded {len(filtered_symbols)} tickers from Nashdaq.")

Successfully loaded 1439 tickers from Nashdaq.


In [10]:
ALPACA_API_KEY = "PKHJPE2YU6BFOQQXIY7L33JI4P"
ALPACA_SECRET_KEY = "E3zKp3YWqym9rdVjTniP8myqb1tWDQSVDd32GZ1Ec3HG"

HEADERS = {
    "APCA-API-KEY-ID":     ALPACA_API_KEY,
    "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY,
}

DATA_URL   = "https://data.alpaca.markets/v2"
SCREEN_URL = "https://data.alpaca.markets/v1beta1"

In [11]:
def get_last_completed_trading_day() -> str:
    """Returns the last completed stock market trading day (YYYY-MM-DD) in Eastern Time."""
    eastern_tz = ZoneInfo("America/New_York")
    now = datetime.now(eastern_tz)
    today = now.date()

    # 0 = Mon, 1 = Tue, 2 = Wed, 3 = Thu, 4 = Fri, 5 = Sat, 6 = Sun
    weekday = today.weekday()

    # Market closes at 4:00 PM ET. 16:30 accounts for data settlement buffer.
    cutoff_time = dtime(16, 30)

    if weekday == 5:  # Saturday -> Friday
        last_day = today - timedelta(days=1)
    elif weekday == 6:  # Sunday -> Friday
        last_day = today - timedelta(days=2)
    elif weekday == 0 and now.time() < cutoff_time:  # Monday before 4:30 PM -> Friday
        last_day = today - timedelta(days=3)
    elif now.time() < cutoff_time:  # Tue-Fri before 4:30 PM -> Yesterday
        last_day = today - timedelta(days=1)
    else:  # Tue-Fri after 4:30 PM -> Today
        last_day = today

    return last_day.strftime("%Y-%m-%d")

In [12]:
def fetch_alpaca_bars(symbols: list, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetches bar data for multiple tickers in batched API calls (Multi-Symbol endpoint)."""
    bars_url = f"{DATA_URL}/stocks/bars"  # Multi-symbol endpoint
    all_bars = []

    # Batch symbols in chunks of 100 to stay safely under URI limits
    batch_size = 100
    symbol_batches = [symbols[i:i + batch_size] for i in range(0, len(symbols), batch_size)]

    for batch in symbol_batches:
        page_token = None
        while True:
            params = {
                "symbols": ",".join(batch),
                "timeframe": "1Day",  # Alpaca format
                "start": start_date,
                "end": end_date,
                "limit": 10000,
                "feed": "iex",
                "adjustment": "raw",
            }
            if page_token:
                params["page_token"] = page_token

            response = requests.get(bars_url, headers=HEADERS, params=params)

            if response.status_code != 200:
                print(f"  ⚠️ API error {response.status_code} — {response.text[:100]}")
                break

            data = response.json()
            bars_dict = data.get("bars", {})  # Map of {"AAPL": [...], "MSFT": [...]}

            for sym, bars in bars_dict.items():
                for b in bars:
                    b["Symbol"] = sym
                    all_bars.append(b)

            page_token = data.get("next_page_token")
            if not page_token:
                break

    if not all_bars:
        return pd.DataFrame()

    df = pd.DataFrame(all_bars)
    df = df.rename(columns={
        "t": "Date", "o": "Open", "h": "High",
        "l": "Low", "c": "Close", "v": "Volume"
    })
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None).dt.normalize()
    keep = ["Date", "Open", "High", "Low", "Close", "Volume", "Symbol"]
    return df[[c for c in keep if c in df.columns]]

In [13]:
def update_stock_history_csv(tickers: list, cache_path: str) -> pd.DataFrame:
    target_end_date = get_last_completed_trading_day()
    print(f"Target end date for dataset: {target_end_date}")

    existing_df = pd.DataFrame()
    max_dates = {}
    updated_dfs = []

    # Load existing history
    if os.path.exists(cache_path):
        print(f"Loading existing history from: {cache_path}")
        try:
            existing_df = pd.read_csv(cache_path)
            existing_df["Date"] = pd.to_datetime(existing_df["Date"])
            max_dates = existing_df.groupby("Symbol")["Date"].max().to_dict()
            print(f"  Loaded {len(existing_df):,} rows for {existing_df['Symbol'].nunique()} symbols")
        except Exception as e:
            print(f"  ⚠️ Could not load existing file: {e} — starting fresh")
            existing_df = pd.DataFrame()
    else:
        print("No existing history file — starting fresh from 2023-01-01")

    if not existing_df.empty:
        updated_dfs.append(existing_df)

    # Determine tickers needing new data
    tickers_to_fetch = []

    for symbol in tickers:
        last_date = max_dates.get(symbol)

        if last_date is not None:
            next_date = last_date + timedelta(days=1)
            # Skip weekends for the start date
            if next_date.weekday() == 5:    # Saturday -> Monday
                next_date += timedelta(days=2)
            elif next_date.weekday() == 6:  # Sunday -> Monday
                next_date += timedelta(days=1)
            start_date = next_date.strftime("%Y-%m-%d")
        else:
            start_date = "2018-06-01"

        if start_date <= target_end_date:
            tickers_to_fetch.append((symbol, start_date))

    if not tickers_to_fetch:
        print("\nAll symbols are up-to-date. No API calls needed.")
        return existing_df

    # Group tickers by required start_date to optimize multi-symbol calls
    start_date_groups = {}
    for sym, s_date in tickers_to_fetch:
        start_date_groups.setdefault(s_date, []).append(sym)

    print(f"\nFetching data for {len(tickers_to_fetch)} out of {len(tickers)} tickers...")

    # Fetch batch data per unique start date
    for s_date, sym_list in start_date_groups.items():
        print(f"\n🚀 Fetching batch of {len(sym_list)} ticker(s) from {s_date} to {target_end_date}:")
        new_data = fetch_alpaca_bars(sym_list, s_date, target_end_date)

        if not new_data.empty:
            updated_dfs.append(new_data)

            # Count bars returned per ticker
            counts = new_data["Symbol"].value_counts()

            # Print status line per individual symbol
            for sym in sym_list:
                bars_count = counts.get(sym, 0)
                if bars_count > 0:
                    print(f"   [{sym:<8}] ✅ {bars_count:,} bars")
                else:
                    print(f"   [{sym:<8}] ⚠️  0 bars returned")
        else:
            print("   ⚠️ No data returned for this entire batch")
            for sym in sym_list:
                print(f"   [{sym:<8}] ⚠️  0 bars returned")

    # Build Master DataFrame
    master_df = (pd.concat(updated_dfs, ignore_index=True)
                 .drop_duplicates(subset=["Date", "Symbol"])
                 .sort_values(["Symbol", "Date"])
                 .reset_index(drop=True))

    # Save and Verify File
    print(f"\nSaving {len(master_df):,} rows to Drive...")
    try:
        master_df.to_csv(cache_path, index=False)
        if os.path.exists(cache_path):
            size_mb = os.path.getsize(cache_path) / 1024 / 1024
            print(f"✅ Saved successfully: {cache_path}")
            print(f"   File size: {size_mb:.2f} MB")
            print(f"   Rows: {len(master_df):,}")
            print(f"   Symbols: {master_df['Symbol'].nunique()}")
        else:
            print("❌ File not found after save — check mount location")
    except Exception as e:
        print(f"❌ Save failed: {e}")

    return master_df

# Execution
stock_history_df = update_stock_history_csv(filtered_symbols, HISTORY_FILE_PATH)

Target end date for dataset: 2026-11-12
Loading existing history from: /content/drive/MyDrive/stocks/stock_history.csv
  Loaded 1,800,637 rows for 1239 symbols

Fetching data for 200 out of 1439 tickers...

🚀 Fetching batch of 200 ticker(s) from 2018-06-01 to 2026-11-12:
  ⚠️ API error 400 — {"message":"invalid symbol: BRK-A"}

  ⚠️ API error 400 — {"message":"invalid symbol: LEN-B"}

   ⚠️ No data returned for this entire batch
   [BAP     ] ⚠️  0 bars returned
   [BAX     ] ⚠️  0 bars returned
   [BB      ] ⚠️  0 bars returned
   [BBD     ] ⚠️  0 bars returned
   [BBDO    ] ⚠️  0 bars returned
   [BBUC    ] ⚠️  0 bars returned
   [BBVA    ] ⚠️  0 bars returned
   [BBY     ] ⚠️  0 bars returned
   [BC      ] ⚠️  0 bars returned
   [BCE     ] ⚠️  0 bars returned
   [BCH     ] ⚠️  0 bars returned
   [BCS     ] ⚠️  0 bars returned
   [BDX     ] ⚠️  0 bars returned
   [BE      ] ⚠️  0 bars returned
   [BEKE    ] ⚠️  0 bars returned
   [BEN     ] ⚠️  0 bars returned
   [BEP     ] ⚠️  0 bar